# ТЗ: ноутбук `02_summarize_smoke.ipynb` — смоук-тест двухэтапной суммаризации

## Контекст

Проект `blind_prophet`: прогноз российских инфляционных ожиданий (инФОМ / ЦБ РФ) на основе новостных Telegram-каналов. Пайплайн: сбор → суммаризация → нейтрализация → прогноз LLM-респондента.

Данное E04 закрыто: зафиксированы параметры RAG-отбора (`config/params.yaml`, секция `rag`), выбрана модель DeepSeek V3.2 (прод) с возможностью отладки на бесплатных OpenRouter-моделях.

Этот ноутбук — смоук-тест **одной конкретной гипотезы**: двухэтапная схема суммаризации (axis-саммари → меташаг) даёт на выходе нарративный таймлайн, из которого LLM-респондент способен извлечь сигнал для ответа на инФОМ-вопрос. Нейтрализация в этот ноутбук не входит — только raw-саммари.

## Цели смоук-теста

1. Убедиться, что двухэтапная схема технически работает end-to-end на реальных данных (calm_2021)
2. Получить материал для ручного ревью: читаемы ли axis-саммари? склеивает ли меташаг кросс-темные узлы? срабатывает ли шлюз-фильтр на слабых осях (труд, зарплаты в calm)?
3. Прогнать три диагностических вопроса (Q1/Q2/Q3) на итоговом саммари и проверить, что:
   - Q1 (идентификация периода) на raw-саммари даёт точность `week` или `month` — это sanity-check что саммари реально отражает период (если даёт `unknown` на raw — саммари слишком обобщённый)
   - Q2 (инФОМ-вопрос про инфляцию от имени персоны) возвращает число, а не отказ
   - Q3 (инфляционные сигналы в тексте) даёт конкретные факты, согласующиеся с направлением Q2

## Архитектура ноутбука

Пять секций:

- **S01** — загрузка периода, top-K по осям, dedup внутри оси
- **S02** — этап 1: 9 параллельных axis-саммари
- **S03** — этап 2: меташаг, склейка в нарративный таймлайн
- **S04** — три диагностических вопроса
- **S05** — сохранение артефактов на диск

## Детали реализации

### Параметры (ячейка config)

```python
PERIOD = "calm_2021"  # ("2021-10-04", "2021-10-18")
TOP_K = 50
DEDUP_THRESHOLD = 0.9
EXCLUDE_CHANNELS = {"prime1"}
LLM_MODEL = os.environ.get("SMOKE_LLM_MODEL", "deepseek/deepseek-chat-v3.1:free")
```

9 RAG-осей — как в `config/params.yaml`.

Клиент — `AsyncOpenAI` c `base_url="https://openrouter.ai/api/v1"`, ключ из `OPENROUTER_API_KEY`.

### S01 — данные

1. Загрузить сообщения периода из БД (`vec_messages` → `processed_messages` → `messages` → `channels`), десериализовать эмбеддинги
2. Отнормировать эмбеддинги (unit-norm)
3. Исключить prime1
4. Посчитать `PERIOD_START = pd.to_datetime(date_from).tz_localize("UTC")` — глобальная переменная, используется в форматтере
5. Эмбеддить 9 осевых запросов (усреднение по axis'у, unit-norm)
6. Для каждой оси — `df.iloc[argsort(-sims)[:TOP_K]]`, получить DataFrame с колонкой `similarity`
7. Dedup внутри оси: из пары с cosine >= 0.9 оставить текст с большей длиной (greedy)

### S02 — axis-саммари

**Форматтер документов — критично для устойчивости к порядку запуска ячеек.** Считает `day_rel` на лету из `subset["date"]` и `PERIOD_START`, НЕ полагается на колонку `day_rel` в subset:

```python
def format_docs_for_prompt(subset: pd.DataFrame) -> str:
    sub = subset.sort_values("date").reset_index(drop=True)
    day_rel = ((sub["date"] - PERIOD_START).dt.total_seconds() // 86400).astype(int) + 1
    lines = []
    for i, row in sub.iterrows():
        text = row["processed_text"].strip().replace("\n", " ")
        lines.append(f"[день {day_rel.iloc[i]} | {row['channel']}] {text}")
    return "\n\n".join(lines)
```

Формат строки документа: `[день N | канал] текст`. Документы сортируются по дате — модель получает хронологический порядок, а не ранжирование по similarity.

**Промпт axis-саммари (system):** роль экономического аналитика, задача — структурированный таймлайн по одной оси, правила:

- относительные даты только (день N, вторая неделя)
- шлюз-фильтр: если документ не относится к заявленной оси — пропускать
- сохранять числа, имена, направления движений, причинно-следственные связки
- группировать по смысловым узлам, одинаковые события схлопывать
- в конце — блок «Ключевые показатели» с числами
- без общих оценок и прогнозов

**User:** `[день N | канал]`-форматированные документы + инструкция «составь таймлайн по оси X».

9 параллельных запросов через `asyncio.gather`. Temperature 0.3. Результат — dict `{axis: {summary, usage, n_docs}}`.

Отдельная ячейка — вывод всех 9 саммари через `IPython.display.Markdown` для ручного ревью.

### S03 — меташаг

**Промпт (system):** роль аналитика, вход — 9 axis-таймлайнов того же периода, задача — единый нарративный таймлайн по событийным узлам.

Правила:
- относительные даты
- группировка по смысловым узлам, не по осям (если один день содержит события из 3+ осей — это один узел)
- в каждом узле: факты / темы которые он затрагивает / причинно-следственные связи с другими узлами
- сохранять все числа и имена собственные из axis-саммари
- 800-1500 слов, связный нарратив по узлам
- в конце — блок «Общий контекст периода», 3-5 предложений без прогнозов
- никаких собственных интерпретаций сверх того что было в axis-саммари

**User:** 9 axis-саммари, конкатенированные с разделителем `=== Ось: X ===`.

Один запрос, temperature 0.3. Вывод — через `Markdown`.

### S04 — три диагностических вопроса

Все три — на выходе меташага. В итоге 3 запроса в этом ноутбуке (в продовом пайплайне будет 3 × 3-5 персон × {raw, neutered}).

**Q1 — идентификация периода.** Temperature 0.0, `response_format={"type": "json_object"}`. Схема ответа:

```json
{
  "precision": "year|month|week|unknown",
  "estimate": "YYYY|YYYY-MM|YYYY-MM-DD..YYYY-MM-DD|null",
  "evidence": "по каким признакам определил"
}
```

**Q2 — персона-респондент.** Temperature 0.7 (живой респондент, не аналитик). System — профиль персоны (женщина, 45, небольшой город, средний доход, следит за ценами в магазинах) + инструкция отвечать исходя ИСКЛЮЧИТЕЛЬНО из текста, без обращения к собственным знаниям о периоде. Вопрос в формулировке инФОМ: «на сколько процентов вырастут цены в России за следующие 12 месяцев». JSON-ответ: `{expected_inflation_pct, reasoning}`.

**Q3 — сигналы в тексте.** Temperature 0.0, JSON:

```json
{
  "signals_up":   [...],
  "signals_flat": [...],
  "signals_down": [...],
  "overall_direction": "up|flat|down|mixed",
  "confidence": "high|medium|low"
}
```

### S05 — сохранение

`../data/smoke/{PERIOD}/`:
- `axis_{axis}.md` × 9
- `meta.md`
- `results.json` — `{period, model, q1, q2, q3, n_docs_by_axis}`

## Что нужно починить агенту

Текущая версия ноутбука содержит баг: в S02 перед `asyncio.gather` был мёртвый цикл `for axis in topk: topk[axis] = topk[axis].merge(...)` , который дописывал `day_rel` в subsets. При повторных запусках он ломал колонку (merge по уже существующему `day_rel` даёт `day_rel_x`/`day_rel_y`), и `format_docs_for_prompt` падал с `KeyError: 'day_rel'`.

**Фикс:**

1. Удалить этот мёртвый цикл полностью (его не должно быть)
2. Переписать `format_docs_for_prompt` так, чтобы он считал `day_rel` на лету из `subset["date"]` и `PERIOD_START`, не полагаясь на колонку `day_rel` в subset'е — код выше

**Дополнительно:**

3. Убрать из S01 строку `df["day_rel"] = ...` — она больше не нужна, `day_rel` теперь считается на лету в форматтере
4. Пройти по ноутбуку и убедиться, что нигде больше нет зависимости от колонки `day_rel` в DataFrame'ах topk/df

## Что НЕ делать

- Не переписывать промпты и логику саммаризации — они валидированы
- Не добавлять нейтрализацию — это следующий этап
- Не расширять до 3-5 персон — для смоука одна
- Не сохранять в БД — только md-файлы и json на диск

## Как проверить что фикс работает

Запуск ноутбука с нуля (Kernel → Restart, Run All) на `PERIOD = "calm_2021"` должен:
- отработать без исключений
- создать 9 `axis_*.md` файлов и один `meta.md`
- создать `results.json` с непустыми q1/q2/q3
- Q1 должен определить период как октябрь 2021 с точностью `month` или `week` — это sanity-check (если `unknown` — проблема не в коде, а в саммари, это уже тема следующего обсуждения)

# 02 — Summarize Smoke Test

Проверка двухэтапной схемы на calm_2021 через бесплатный API (Qwen).

**Схема:**
1. Axis-саммари: 9 параллельных запросов, по top-K доков на ось → структурированный axis-таймлайн
2. Меташаг: склейка 9 axis-саммари в нарративный таймлайн по смысловым узлам

**Проверяем:**
- Q1 — идентификация периода (точность: год / месяц / неделя / не могу)
- Q2 — инФОМ-вопрос про ожидание цен (персона-респондент)
- Q3 — какие сигналы в тексте указывают на рост/стабильность/снижение цен

На этом этапе — ТОЛЬКО raw саммари, без нейтрализации. Цель — убедиться что двухэтапная схема даёт читаемый и информативный выход.

## Setup

In [1]:
import asyncio
import json
import os
import struct
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openai import AsyncOpenAI

sys.path.insert(0, str(Path.cwd().parent))

from amnesiac.store.db import get_connection
from amnesiac.store.embeddings import embed_query

In [2]:
# --- Эксперимент ---
PERIODS = {
    "calm_2021":   ("2021-10-04", "2021-10-18"),
    "calm_2023":   ("2023-04-17", "2023-05-01"),
    "shock_covid": ("2020-04-06", "2020-04-20"),
    "shock_war":   ("2022-03-07", "2022-03-21"),
}
PERIOD = "calm_2021"

# --- RAG ---
TOP_K = 50
DEDUP_THRESHOLD = 0.9
EXCLUDE_CHANNELS = {"prime1"}

# --- LLM ---
# OpenRouter: Qwen3 (бесплатная для smoke), DeepSeek для прода — сменим строкой ниже
LLM_MODEL = os.environ.get("SMOKE_LLM_MODEL", "deepseek/deepseek-v3.2")
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
# ключ в ~/.bashrc или .env
OPENROUTER_KEY = "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"  #os.environ["OPENROUTER_API_KEY"]

# --- Прочее ---
DB_PATH = Path("../data/db/blind_prophet.db")
_EMBEDDING_DIM = 1536

RAG_QUERIES = {
    "дкп": ["решение по ключевой ставке", "денежно-кредитная политика Банка России"],
    "инфляция": ["рост потребительских цен", "инфляция в России"],
    "продовольствие": ["цены на продукты питания"],
    "курс": ["курс рубля к доллару", "ослабление рубля"],
    "тарифы": ["повышение тарифов ЖКХ", "цены на бензин"],
    "зарплаты": ["рост зарплат", "реальные доходы населения"],
    "труд": ["дефицит кадров в России", "безработица в России"],
    "кризис": ["экономический кризис в России", "экономическая неопределённость", "санкции против России"],
    "бюджет": ["бюджетные расходы", "дефицит бюджета"],
}

client = AsyncOpenAI(base_url=OPENROUTER_BASE, api_key=OPENROUTER_KEY)
print(f"Period: {PERIOD}  |  Model: {LLM_MODEL}")

Period: calm_2021  |  Model: deepseek/deepseek-v3.2


## S01 — Загрузка и top-K по осям

Воспроизводим логику E04/S03: exclude prime1, top-K на ось, dedup пар >= 0.9 (оставляем более длинный текст).

In [3]:
def load_period(conn, date_from: str, date_to: str) -> pd.DataFrame:
    sql = """
        SELECT vm.message_id AS message_id, ch.username AS channel, m.date AS date,
               pm.processed_text AS processed_text, vm.embedding AS embedding_blob
        FROM vec_messages vm
        JOIN processed_messages pm ON pm.id = vm.message_id
        JOIN messages m            ON m.id = pm.message_id
        JOIN channels ch           ON ch.id = m.channel_id
        WHERE m.date BETWEEN ? AND ? AND pm.is_valid = 1
        ORDER BY m.date
    """
    rows = conn.execute(sql, (date_from, date_to)).fetchall()
    records = []
    for mid, ch, d, text, blob in rows:
        vec = np.array(struct.unpack(f"<{_EMBEDDING_DIM}f", blob), dtype=np.float32)
        records.append({
            "message_id": mid, "channel": ch,
            "date": datetime.fromisoformat(d) if isinstance(d, str) else d,
            "processed_text": text, "embedding": vec,
        })
    return pd.DataFrame(records, columns=["message_id", "channel", "date", "processed_text", "embedding"])


conn = get_connection(str(DB_PATH))
date_from, date_to = PERIODS[PERIOD]
df = load_period(conn, date_from, date_to)

# normalize embeddings
X = np.stack(df["embedding"].values)
X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-9, None)
df["embedding"] = list(X)

# exclude prime1
df = df[~df["channel"].isin(EXCLUDE_CHANNELS)].reset_index(drop=True)

# period start for relative days
PERIOD_START = pd.to_datetime(date_from).tz_localize("UTC")

print(f"Loaded: {len(df)} docs (after prime1 exclusion)")


Loaded: 5494 docs (after prime1 exclusion)


In [4]:
async def embed_axes(queries: dict) -> dict:
    result = {}
    for axis, texts in queries.items():
        vecs = await asyncio.gather(*(embed_query(t) for t in texts))
        mean = np.mean(vecs, axis=0).astype(np.float32)
        mean = mean / np.clip(np.linalg.norm(mean), 1e-9, None)
        result[axis] = mean
    return result

query_vectors = await embed_axes(RAG_QUERIES)
print(f"Axes embedded: {list(query_vectors.keys())}")

Axes embedded: ['дкп', 'инфляция', 'продовольствие', 'курс', 'тарифы', 'зарплаты', 'труд', 'кризис', 'бюджет']


In [5]:
def top_k_by_axis(df: pd.DataFrame, query_vectors: dict, k: int) -> dict:
    """top-K по каждой оси, отсортировано по similarity desc."""
    X = np.stack(df["embedding"].values)
    out = {}
    for axis, q in query_vectors.items():
        sims = X @ q
        top_idx = np.argsort(-sims)[:k]
        sub = df.iloc[top_idx].copy()
        sub["similarity"] = sims[top_idx]
        out[axis] = sub.reset_index(drop=True)
    return out


def dedup_within(subset: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Убираем near-duplicates внутри оси: из пары оставляем более длинный текст.
    Greedy: идём по парам выше threshold и помечаем короткий на выброс.
    """
    if len(subset) < 2:
        return subset.reset_index(drop=True)
    X = np.stack(subset["embedding"].values)
    sim = X @ X.T
    np.fill_diagonal(sim, 0.0)
    text_len = subset["processed_text"].str.len().values
    drop = set()
    n = len(subset)
    for i in range(n):
        if i in drop:
            continue
        for j in range(i + 1, n):
            if j in drop:
                continue
            if sim[i, j] >= threshold:
                loser = j if text_len[i] >= text_len[j] else i
                drop.add(loser)
                if loser == i:
                    break
    keep = [i for i in range(n) if i not in drop]
    return subset.iloc[keep].reset_index(drop=True)


topk = top_k_by_axis(df, query_vectors, TOP_K)
topk = {axis: dedup_within(sub, DEDUP_THRESHOLD) for axis, sub in topk.items()}

for axis, sub in topk.items():
    print(f"  {axis:<16} {len(sub):>3} docs (after dedup)")

  дкп               49 docs (after dedup)
  инфляция          46 docs (after dedup)
  продовольствие    43 docs (after dedup)
  курс              37 docs (after dedup)
  тарифы            35 docs (after dedup)
  зарплаты          46 docs (after dedup)
  труд              50 docs (after dedup)
  кризис            50 docs (after dedup)
  бюджет            47 docs (after dedup)


## S02 — Этап 1: axis-саммари

Девять параллельных запросов. Вход — отсортированные по дате документы оси. Выход — структурированный axis-таймлайн с относительными датами.

Промпт явно разрешает выкидывать нерелевантные документы (шлюз-фильтр).

In [6]:
AXIS_SYSTEM = """Ты — экономический аналитик. Твоя задача — составить структурированный таймлайн событий по одной конкретной тематической оси из потока новостных сообщений.

Правила:

1. **Относительные даты везде.** Используй «день 1», «день 5-7», «вторая неделя». Это касается:
   - дат публикации новостей (маркер `[день N | канал]` на входе)
   - **дат событий ВНУТРИ самих новостей** — если в тексте написано «с 18 октября АвтоВАЗ повысил цены» или «инфляция за неделю 21-27 сентября составила 7.26%», ты должен переписать это в относительных терминах («в конце периода производитель повысил цены», «по данным за неделю в конце предыдущего месяца инфляция составила 7.26%») или убрать дату вообще, если она не несёт смысла.
   - **никаких упоминаний месяцев и лет** (сентябрь, октябрь, 2021) — ни в каком виде.
   - **никаких числовых дат** (21-27, 18.10 и т.п.).

2. **Релевантность оси — функциональная, а не тематическая.** Сохраняй только те сообщения, которые содержательно влияют на прогноз потребительских цен и инфляционных ожиданий. Для оси «{axis}» это означает:
   - ось про макроэкономический показатель, денежную политику, цены или доходы — сохраняй
   - ось про организационную культуру, HR-исследования, конфликты на работе, карьерные истории, бытовые частности — пропускай, даже если тема формально совпадает

3. Сохраняй числовые показатели (проценты, значения курса, ставки, объёмы) и имена собственные — они важны для последующего анализа.
4. Сохраняй направления движений: подорожал/подешевел, вырос/упал, повысил/понизил.
5. Фиксируй причинно-следственные связки там, где они явно прослеживаются.
6. Формат: структурированный список событий по смысловым узлам, а не по каждому сообщению в отдельности. Одинаковые/близкие события схлопывай в один пункт.
7. В конце — краткий блок «Ключевые показатели»: числа и значения, встречающиеся в текстах (без дат).
8. Не добавляй общих оценок, прогнозов или мета-комментариев. Только факты из текстов.

Ось: {axis}
Период: 14 дней
"""

AXIS_USER_TEMPLATE = """Новостные сообщения (отсортированы по времени, формат [день N | канал] текст):

{docs}

Составь таймлайн по оси «{axis}»."""


def format_docs_for_prompt(subset: pd.DataFrame) -> str:
    sub = subset.sort_values("date").reset_index(drop=True)
    day_rel = ((sub["date"] - PERIOD_START).dt.total_seconds() // 86400).astype(int) + 1
    lines = []
    for i, row in sub.iterrows():
        text = row["processed_text"].strip().replace("\n", " ")
        lines.append(f"[день {day_rel.iloc[i]} | {row['channel']}] {text}")
    return "\n\n".join(lines)

In [7]:
async def summarize_axis(axis: str, subset: pd.DataFrame) -> dict:
    docs = format_docs_for_prompt(subset)
    messages = [
        {"role": "system", "content": AXIS_SYSTEM.format(axis=axis)},
        {"role": "user",   "content": AXIS_USER_TEMPLATE.format(axis=axis, docs=docs)},
    ]
    resp = await client.chat.completions.create(
        model=LLM_MODEL, messages=messages, temperature=0.3,
    )
    return {
        "axis": axis,
        "summary": resp.choices[0].message.content,
        "usage": resp.usage.model_dump() if resp.usage else None,
        "n_docs": len(subset),
    }


axis_summaries = await asyncio.gather(*(
    summarize_axis(axis, subset) for axis, subset in topk.items()
))
axis_summaries = {r["axis"]: r for r in axis_summaries}

for axis, r in axis_summaries.items():
    u = r["usage"] or {}
    print(f"  {axis:<16} {r['n_docs']:>2} docs  |  tokens in/out: {u.get('prompt_tokens', '?')}/{u.get('completion_tokens', '?')}")

  дкп              49 docs  |  tokens in/out: 4611/1577
  инфляция         46 docs  |  tokens in/out: 4424/1846
  продовольствие   43 docs  |  tokens in/out: 4183/1444
  курс             37 docs  |  tokens in/out: 3553/1168
  тарифы           35 docs  |  tokens in/out: 2951/1134
  зарплаты         46 docs  |  tokens in/out: 4196/997
  труд             50 docs  |  tokens in/out: 4626/1286
  кризис           50 docs  |  tokens in/out: 3955/958
  бюджет           47 docs  |  tokens in/out: 4010/1211


In [8]:
from IPython.display import Markdown, display

for axis in RAG_QUERIES:
    display(Markdown(f"### Ось: **{axis}**\n\n{axis_summaries[axis]['summary']}\n\n---"))

### Ось: **дкп**

### Таймлайн по оси «дкп»

**Ранний период (дни 1-3)**
*   Цены на энергоносители демонстрируют значительный рост: нефть марки Brent превысила $81 за баррель, а фьючерсы на газ в Европе обновили рекорд, достигнув $1427 за тысячу кубометров. Рублевая цена российской нефти Urals также обновила исторический максимум, превысив 5.87 тыс. рублей.
*   Банк России сигнализирует о высокой вероятности пересмотра прогноза по инфляции на год в сторону повышения с текущего диапазона 5.7-6.2%, ссылаясь на рост цен в конце предыдущего месяца выше ожиданий. Регулятор отмечает озабоченность влиянием высоких цен на энергоносители в Европе на импортируемую инфляцию.
*   ЦБ сохраняет сигнал о возможности дальнейшего повышения ключевой ставки (сейчас 6.75%) на ближайших заседаниях, одновременно заявляя, что разговоры о её снижении преждевременны, а двузначных значений ставки не потребуется. Аналитики Saxo Bank не исключают повышения ставки до 7.25% в текущем году.
*   Всемирный банк улучшает прогноз роста ВВП России на год до 4.3% (с 3.2%) в связи с восстановлением внутреннего спроса и высокими ценами на энергоносители.
*   Банки нарастили выдачу розничных кредитов на 2.5% до 1.25 трлн рублей, прирост обеспечен ипотекой, в то время как выдачи наличными и автокредиты сократились. Средний чек одной валютообменной операции вырос на 19% до 73.35 тыс. рублей.
*   Минфин ожидает лишь краткосрочного роста доходов бюджета от рекордных цен на газ и обсуждает возможность повышения порога для инвестиций из Фонда национального благосостояния (ФНБ) до 10% ВВП для бюджетной стабильности, что вызывает дискуссии.

**Середина периода (дни 4-9)**
*   Курсы доллара и евро на Московской бирже снижаются: доллар опускается ниже 72 рублей, а евро — до 83 рублей.
*   Минэкономразвития существенно повышает прогноз по инфляции на год до 7.4% (с 5.8%), основной причиной называя ускорение продовольственной инфляции. Официальные лица связывают рост цен с ситуацией на внешних рынках и посткризисной волатильностью.
*   Банк России публикует данные: годовая инфляция в конце предыдущего месяца составила 7.4%, а по данным на начало текущего периода ускорилась до 7.63%. Регулятор отмечает, что разовые проинфляционные факторы могут дольше держать инфляционные ожидания повышенными.
*   ЦБ заявляет, что, несмотря на ожидаемое замедление темпов роста инфляции в последнем квартале года, дефицит кадров в экономике может привести к возникновению инфляционной спирали. При этом регулятор ожидает замедления инфляции до 4-4.5% в следующем году.
*   Правительство предпринимает меры для сдерживания цен: поддерживается продление льготы на ввоз сахара до конца года, а ФАС сообщает о снижении предельных оптовых надбавок на лекарства в Москве в среднем на 3.7%. Минпромторг не видит предпосылок для резкого роста цен на продукты.
*   ЦБ продолжает меры по охлаждению рынка потребкредитования, намереваясь обновить методику расчета долговой нагрузки заемщика, что, по мнению банкиров, может снизить выдачу кредитов.

**Поздний период (дни 10-14)**
*   Министр финансов заявляет о рисках развития стагфляционного сценария в мировой экономике.
*   Действия правительства по сдерживанию цен на продукты признаются эффективными, но требующими долгосрочных мер. Премьер-министр поручает координировать эту работу.
*   Несмотря на рост ключевой ставки, ставки по розничным кредитам остаются стабильными (средняя полная стоимость кредита (ПСК) во втором квартале — 13.6%), что объясняется готовностью банков жертвовать маржой для наращивания портфелей.
*   ЦБ совместно с Минфином работает над поправками для уточнения порядка расчета полной стоимости кредита, чтобы сделать реальные расходы граждан более ясными.
*   Минфин объявляет о повышении экспортной пошлины на нефть на $8.4, до $71.2 за тонну, с начала следующего месяца.
*   Объявлено о повышении с начала следующего года минимального размера пособия по уходу за первым ребенком на 5.8% до 7 493 рублей, максимального — до 31 282 рублей.

---
### Ключевые показатели
*   **Цены на энергоносители:** Brent >$81; газ в Европе: $1427/тыс.куб.м; Urals: >5.87 тыс. руб.
*   **Инфляция:** прогноз Минэкономразвития на год: 7.4%; фактическая годовая инфляция на конец предыдущего месяца: 7.4%; на начало текущего периода: 7.63%; прогноз ЦБ на следующий год: 4-4.5%.
*   **Ключевая ставка ЦБ:** 6.75% (возможное повышение до 7.25% по оценкам Saxo Bank).
*   **ВВП:** прогноз роста на год (Всемирный банк): 4.3%.
*   **Курсы валют:** доллар <72 руб.; евро: 83 руб.
*   **Кредитование:** выдачи розничных кредитов: 1.25 трлн руб. (+2.5%); средняя ПСК: 13.6%.
*   **Бюджет:** потенциальные дополнительные доходы от высоких цен на нефть: >3.2 трлн руб.
*   **Социальные выплаты:** пособие по уходу за первым ребенком: мин. 7 493 руб., макс. 31 282 руб. (с начала следующего года).

---

### Ось: **инфляция**

### Таймлайн по оси «Инфляция»

**Первая неделя**

*   **День 1-2:** Курс доллара и евро на Московской бирже вырос. Цена газа в Европе обновила исторический максимум, превысив $1450 за тыс. куб. м.
*   **День 2:** Официальные лица заявляют, что инфляция близка к пику, после которого вероятно снижение. Банк России назвал высоко вероятным пересмотр прогноза по инфляции на текущий год в сторону повышения. По данным Минэкономразвития, годовая инфляция за неделю в конце предыдущего месяца ускорилась до 7,26%.
*   **День 3:** Росстат сообщает, что инфляция в месячном выражении ускорилась до 0,6%, в годовом выражении — до 7,4%. Наибольший вклад внесли продукты (+9,2% год к году), особенно плодоовощная продукция (+15,2%). По непродовольственным товарам +8,1%, услугам +4,2%. Банк России намекнул, что повысит прогноз по инфляции на текущий год, так как цены росли выше ожиданий. ЦБ по-прежнему рассчитывает на снижение темпов роста инфляции в IV квартале. Банк России отмечает, что рост цен на энергоносители в Европе вызывает озабоченность и может сказаться на импортируемой инфляции.
*   **День 3:** Отмечается рост цен на ряд товаров и услуг: средний чек валютообменной операции вырос на 19%; бургеры в среднем подорожали на 10,4% в годовом выражении, картошка фри – на 18%, газированные напитки – на 5%; цены на «вторичку» в Москве снова растут после снижения в предыдущем месяце.
*   **День 4:** Курс доллара и евро на Московской бирже снижается (евро опустился ниже 83 рублей). Минпромторг не видит предпосылок для резкого роста цен на продукты и новых соглашений о стабилизации цен. ФАС сообщила, что предельные оптовые надбавки к ценам на лекарства в Москве снижены в среднем на 3,7%. Эксперты прогнозируют, что сигареты до конца года подорожают примерно на 10 рублей за пачку.

**Вторая неделя**

*   **День 5:** Минэкономразвития ожидает рост цен на непродовольственные товары на 7,2% к концу года. Стоимость продовольственных товаров повысится на 5,3%. Первый вице-премьер Белоусов заявил, что инфляция к концу года может оказаться на уровне 6,5-7% или ниже, и цены на продукты после аномального роста должны скоро выправиться.
*   **День 9:** Минэкономразвития повысило прогноз инфляции в текущем году до 7,4% (с 5,8%). Основной вклад в рост цен вносит продовольственная инфляция, главными причинами стали нетипичное удорожание овощей и фруктов, а также ускорение роста цен на мясо. Прогноз инфляции на следующие два года остается неизменным — 4% на конец года. Официальные лица связывают превышение прогнозного уровня инфляции с посткризисной волатильностью на международных рынках и ситуацией на глобальных рынках энергоресурсов.
*   **День 10:** АвтоВАЗ повышает цены на весь модельный ряд в среднем на 1,8% в конце периода. Годовая инфляция возросла до 7,4%, большую роль сыграла динамика цен на плодоовощную продукцию. Банк России отмечает, что разовые проинфляционные факторы могут дольше держать инфляционные ожидания повышенными, а дефицит кадров в экономике может привести к возникновению инфляционной спирали. ЦБ ожидает замедление инфляции до 4-4,5% в следующем году.
*   **День 10:** Недельная инфляция замедлилась до 0,22%. Годовая инфляция ускорилась до 7,63%. Средние цены на бензин за неделю выросли на 6 коп. Рост цен на бензин по итогам года планируется на уровне, близком к инфляции. С начала года бензин подорожал на 6,2%.
*   **День 10:** Лидер роста цен среди продуктов питания сменился: огурцы за неделю прибали в стоимости 10,9%, помидоры - 9,7%. Картофель подорожал на 4,4%, яйца - на 3,3%, курица - на 1,5%, гречка - на 0,9%. Яблоки подешевели на 1,4%, морковь - на 1%, лук - на 0,3%. С начала года огурцы -38,1%, томаты -16,8%, яблоки -2,1%, яйца -0,8%, картофель +38,9%, морковь +31,4%, курица +22,6%, лук +21,1%, гречка +12,9%.
*   **День 11:** Прожиточный минимум на душу населения в Москве в следующем году увеличится до 18 714 руб. (в текущем году составил 18 029 руб.).
*   **День 12:** Экспортная пошлина на нефть повысится на $8,4 и составит $71,2 за тонну с начала следующего месяца. Курс доллара опустился ниже 71 руб.

### Ключевые показатели
*   Инфляция (годовая): 7.26%, 7.4%, 7.63%.
*   Инфляция (месячная): 0.6%.
*   Инфляция (недельная): 0.22%, 0.26%.
*   Прогнозы инфляции: 5.7-6.2%, 6.5-7%, 7.4%.
*   Курсы валют: доллар 72.97 руб., 71 руб.; евро 84.51 руб., 83 руб.
*   Цена газа: $1450 за тыс. куб. м.
*   Цены на продукты (год к году): +9.2%; плодоовощная +15.2%.
*   Цены на непродовольственные товары (год к году): +8.1%.
*   Цены на услуги (год к году): +4.2%.
*   Прогноз роста цен к концу года: непродовольственные товары +7.2%, продовольственные +5.3%.
*   Средний чек валютообменной операции: 73.35 тыс. руб. (рост на 19%).
*   Цены на бургеры: 311 руб. (рост 10.4% год к году); картошка фри: 111 руб. (рост 18%); газировка: 72.3 руб. (рост 5%).
*   Цены на бензин: рост 6 коп. за неделю; рост 6.2% с начала года.
*   Прожиточный минимум в Москве: 18 029 руб. (2021), 18 714 руб. (2022).
*   Экспортная пошлина на нефть: $71.2 за тонну.
*   Средняя ставка ПСК по розничным кредитам: 13.6%.

---

### Ось: **продовольствие**

### Таймлайн по оси «продовольствие»

**Ранний период (дни 1-4)**
*   Сетевые ритейлеры снизили цены на некоторые продукты в отдельных регионах после вмешательства регулятора.
*   По данным за предыдущий месяц, среди продуктов лидерами роста цен стали помидоры (+24,9%), огурцы (+17,3%), апельсины (+12,9%), чеснок (+6,5%), яйца (+5,7%), бананы (+3,4%), куры (+3,2%), говядина (+3%). При этом подешевели свекла (-13,7%), виноград (-12,6%), морковь (-10,3%), яблоки (-5,7%), капуста (-3,6%), лук (-2,8%).
*   По данным исследования, с начала года в России значительно подорожали позиции фастфуда: бургеры (на 10,4%), картофель фри (на 18%), газированные напитки (на 5%). Эксперты связывают это с ростом стоимости логистики, бензина и ингредиентов (мясо и помидоры подорожали за год на 15-20%).
*   Мировые цены на продовольствие достигли рекордного за 10 лет уровня.
*   Банк России выразил озабоченность ростом цен на энергоносители в Европе, отметив их потенциальное влияние на импортируемую инфляцию.
*   Министерство промышленности и торговли заявило, что не видит предпосылок для резкого роста цен на продукты.

**Середина периода (дни 5-9)**
*   Министерство экономического развития ожидает рост цен на продовольственные товары на 5,3% к концу года.
*   Президент заявил, что в стране собран хороший урожай, практически полностью обеспечивающий потребности по основным группам продовольствия.
*   В Европейском союзе ожидают рост цен на продукты питания из-за увеличения стоимости энергии, которая ведет к удорожанию удобрений.
*   Президент отметил, что рост цен на продовольствие в России превысил 7,5%, частично связав это с ситуацией на глобальных рынках энергоресурсов.

**Поздний период (дни 10-12)**
*   Динамика цен на отдельные продукты изменилась: за неделю огурцы подорожали на 10,9%, помидоры — на 9,7%, картофель — на 4,4%, яйца — на 3,3%, курица — на 1,5%, гречка — на 0,9%. При этом подешевели яблоки (на 1,4%), морковь (на 1%), лук (на 0,3%).
*   С начала года значительный рост цен показали картофель (+38,9%), морковь (+31,4%), курица (+22,6%), лук (+21,1%), гречка (+12,9%). Огурцы и томаты за тот же период подешевели.
*   Премьер-министр заявил, что меры по сдерживанию цен на продукты оказались эффективными, но для долгосрочной стабилизации требуются дополнительные шаги. Он поручил координировать эту работу.
*   Минсельхоз прорабатывает меры поддержки личных подсобных хозяйств и новые льготы для производителей мяса.
*   Сетевой ритейлер снизил цены на морковь и капусту в одном из регионов после предупреждения регулятора.

### Ключевые показатели
*   **Изменения цен на продукты (за месяц):** помидоры +24,9%, огурцы +17,3%, апельсины +12,9%, чеснок +6,5%, яйца +5,7%, бананы +3,4%, куры +3,2%, говядина +3%; свекла -13,7%, виноград -12,6%, морковь -10,3%, яблоки -5,7%, капуста -3,6%, лук -2,8%.
*   **Изменения цен на продукты (за неделю):** огурцы +10,9%, помидоры +9,7%, картофель +4,4%, яйца +3,3%, курица +1,5%, гречка +0,9%; яблоки -1,4%, морковь -1%, лук -0,3%.
*   **Изменения цен с начала года (продукты):** картофель +38,9%, морковь +31,4%, курица +22,6%, лук +21,1%, гречка +12,9%; огурцы -38,1%, томаты -16,8%, яблоки -2,1%, яйца -0,8%.
*   **Изменения цен на фастфуд с начала года:** бургеры +10,4% (до 311 руб.), картошка фри +18% (до 111 руб.), газированные напитки +5% (до 72,3 руб.).
*   **Прогноз Минэкономразвития:** рост цен на продовольственные товары к концу года — 5,3%.
*   **Заявление президента:** рост цен на продовольствие превысил 7,5%.
*   **Индекс ФАО:** мировые цены на продовольствие достигли рекордного за 10 лет уровня.
*   **Средние чеки (Covindex):** антисептики 267 руб. (+6%), перчатки 143 руб. (+2%), медицинские маски 65 руб., туалетная бумага 75 руб., гречка 90 руб., лимоны 58 руб. (-5%), кусковое мыло 88 руб. (-2%).
*   **Прожиточный минимум в Москве:** в следующем году составит 18 714 руб. (в текущем — 18 029 руб.).

---

### Ось: **курс**

### Таймлайн по оси «курс»

**Начало периода**
*   Курс доллара на Московской бирже вырос до 72,97 рублей, курс евро — до 84,51 рублей.
*   Цены на нефть (WTI и Brent) достигли многолетних максимумов, превысив $77 и $81 за баррель соответственно.
*   Цены на газ в Европе начали резкий рост, превысив $1250 за тысячу кубометров.

**День 2**
*   Цены на газ в Европе продолжили рекордный рост, достигнув $1450 за тысячу кубометров (рост более 20% за сутки).
*   Курс евро на Московской бирже опустился ниже 84 рублей.
*   Нефть марки Brent продолжила рост, превысив $83 за баррель.

**День 3**
*   Рублевая цена российской нефти Urals обновила исторический максимум, достигнув 5,87 тыс. рублей. С конца предыдущего месяца стоимость выросла более чем на 23%.
*   Цены на энергетический уголь в Европе преодолели отметку $300 за тонну, обновив исторический рекорд.
*   Стоимость биткойна выросла более чем на 10%, превысив $54,9 тыс.

**День 4**
*   Цена фьючерсов на газ в Европе резко упала почти на 15%, до $1110 за тысячу кубометров.
*   Курс доллара на Московской бирже опустился ниже 72 рублей, курс евро упал до 83 рублей (впервые за более чем год).
*   Эксперты высказали мнение, что рубль может укрепиться к доллару до уровня около 71,55 рубля, но курса ниже 71 рубля ожидать не стоит.

**День 5**
*   Финансовые эксперты ожидали, что на следующей неделе курс доллара прибавит около 50 копеек, а евро — 40 копеек. Прогнозировалось, что цена на нефть марки Brent останется немного ниже $82,5 за баррель.

**День 8**
*   Цена нефти марки Brent превысила $84 за баррель, а WTI достигла $80,52 за барреля (рекорд за последние семь лет).

**День 9**
*   Стоимость алюминия на Лондонской бирже металлов поднялась выше $3100 за тонну.

**День 10**
*   Президент РФ Владимир Путин заявил, что доллар подрывает свои позиции в качестве резервной валюты, но Россия не заинтересована в полном уходе от расчетов в нем.

**День 11**
*   Международные резервы РФ выросли на $3,5 млрд, достигнув $615,4 млрд.
*   Глава ЦБ Эльвира Набиуллина заявила, что хранит свои сбережения в рублях, считая это выгодным и этичным.

**День 12**
*   Цена нефти марки Brent выросла, достигнув $85,03 за баррель.
*   Экспортная пошлина на нефть в РФ будет повышена на $8,4 и составит $71,2 за тонну.
*   Цена на алюминий продолжила рост, превысив $3,2 тыс. за тонну.
*   Цена на авиакеросин на бирже обновила рекорд, превысив 56,6 тыс. рублей за тонну.
*   **Курс доллара на Московской бирже опустился ниже 71 рубля, курс евро составил 82,33 рубля.**

---
### Ключевые показатели
*   **Курсы валют:** доллар: 72.97, 71.79, 71.55, 71.78, 70.93; евро: 84.51, 83, 82.33.
*   **Цены на энергоносители:** нефть Brent: $81, $83, $82.5, $84, $85.03; нефть WTI: $77, $80.52; газ: $1250, $1427, $1450, $1110, $1163; уголь: $300.
*   **Цены на металлы:** алюминий: $3100, $3200.
*   **Другие товары:** рублевая цена нефти Urals: 5.87 тыс. руб.; авиакеросин: 56.6 тыс. руб./тн; биткойн: $54.9 тыс.
*   **Макропоказатели:** Международные резервы РФ: $615.4 млрд; экспортная пошлина на нефть: $71.2/тн.

---

### Ось: **тарифы**

### Таймлайн по оси «тарифы»

**Первая половина периода**

*   **Начало периода:** Цены на энергоносители на мировых рынках демонстрируют сильный рост. Цена на газ в Европе достигает новых исторических максимумов, последовательно превышая уровни в $1230, $1350, $1427, $1450, $1550 и $1600 за тысячу кубометров. Параллельно растут цены на нефть (WTI превышает $77 за баррель) и уголь (превышает $300 за тонну, обновляя исторический рекорд).
*   **В середине первой недели:** После публичных заявлений о готовности помочь стабилизировать рынок, цена на газ в Европе резко снижается. За сутки она падает почти на 15%, опускаясь ниже $1250, а затем и ниже $1000 за тысячу кубометров. В это же время ФАС сообщает о снижении предельных оптовых надбавок на лекарства в Москве в среднем на 3,7%.
*   **Конец первой недели:** Минэкономразвития публикует ожидания по росту потребительских цен к концу года: на непродовольственные товары — на 7,2%, на продовольственные — на 5,3%.

**Вторая половина периода**

*   **Начало второй недели:** На мировых рынках возобновляется рост цен на сырье. Цена на алюминий превышает $3000 за тонну, а нефть марки Brent — $84 за баррель, что является многолетним максимумом. В ЕС официально заявляют о риске роста цен на продовольствие из-за увеличения стоимости энергии и удобрений.
*   **Середина второй недели:** Производитель автомобилей в России объявляет о повышении цен на весь модельный ряд в среднем на 1,8%. Одновременно с этим официальные лица констатируют, что цены на продовольствие в стране уже выросли более чем на 7,5%, частично связывая это с ситуацией на глобальных энергорынках. Замглавы Минэнерго заявляет, что рост цен на бензин по итогам года планируется на уровне, близком к инфляции.
*   **Конец второй недели:** Динамика на энергорынках остается волатильной: цена на газ в Европе снова превышает $1200 за тысячу кубометров, но затем опускается ниже $1180. Нефть продолжает дорожать, цена Brent достигает $85,03 за баррель. При этом глава одной из нефтяных компаний в России заявляет о начале снижения цен на топливо на АЗС. Минфин сообщает о повышении экспортной пошлины на нефть на $8,4. Цена на авиакеросин на бирже обновляет рекорд, превысив 56,6 тыс. рублей за тонну.

---

### Ключевые показатели
*   **Газ в Европе:** цены от ~$1000 до исторических максимумов ~$1600 (отдельное упоминание о пике в $1920) за тыс. куб. м.
*   **Нефть:** WTI — $77, $80.52 за баррель; Brent — $84, $85.03 за баррель.
*   **Уголь:** более $300 за тонну.
*   **Алюминий:** более $3000 за тонну.
*   **Бензин Аи-95:** ниже 60 тыс. рублей за тонну на бирже.
*   **Бензин Аи-92:** 57,47 тыс. рублей за тонну на бирже.
*   **Авиакеросин:** более 56,6 тыс. рублей за тонну.
*   **Экспортная пошлина на нефть в РФ:** повышение на $8.4, до $71.2 за тонну.
*   **Прогнозы роста цен в РФ (Минэкономразвития):** продовольствие +5.3%, непродовольственные товары +7.2% к концу года.
*   **Рост цен на продукты быстрого питания (в годовом выражении):** бургеры +10.4% (до 311 руб.), картошка фри +18% (до 111 руб.), газировка +5% (до 72.3 руб.).
*   **Сигареты:** прогнозируемый рост на ~10 руб. за пачку (со ~135.8 до 140-145 руб.).
*   **Лекарства:** снижение предельных оптовых надбавок в Москве в среднем на 3.7%.
*   **Автомобили Lada:** среднее повышение цен на 1.8%.

---

### Ось: **зарплаты**

### Таймлайн по оси «Зарплаты»

**Первая неделя**
*   В начале периода публикуются данные о средней зарплате в одном из крупных городов, составляющей 73,9 тыс. рублей, при этом отмечается падение ежемесячных доходов работающих жителей.
*   Власти сообщают о снижении числа официально зарегистрированных безработных до 880 тыс. человек.
*   Появляются результаты опроса, согласно которому значительная часть россиян (около 40%) готова отказаться от нескольких выходных в месяц ради повышения зарплаты, а 12% респондентов готовы работать без отдыха за доход в 128,4 тыс. рублей в месяц.
*   Публикуется информация о вакансиях с очень высокими зарплатами (до 500, 400 и 250 тыс. рублей) в столице для узкоспециализированных профессий.
*   Статистическое ведомство анонсирует планы по уточнению данных о денежных доходах населения за счет включения информации о состоянии брокерских счетов, что может скорректировать оценку уровня благосостояния.
*   Объявлено о повышении максимального размера пособия по безработице в следующем году до 12 792 руб.

**Вторая неделя**
*   На высшем уровне заявляется, что низкие доходы миллионов граждан являются главной угрозой стабильному развитию. Власти фиксируют проблемы в экономике и выражают обеспокоенность из-за падения доходов населения.
*   В связи с этим дано поручение разработать дополнительные меры поддержки граждан в условиях повышенной инфляции, а также объявлено о переходе на индексацию материнского капитала по фактической, а не прогнозной инфляции.
*   Публикуются данные о крайне низкой средней зарплате в одном из регионов (около 32 тыс. рублей, что на 60% ниже среднероссийского уровня). Глава региона отмечает, что даже такая сумма является «большим подарком» для среднестатистического жителя, и указывает на рост числа людей за чертой бедности (почти полмиллиона, увеличение примерно на 18% за 5 лет).
*   Уровень безработицы в стране снижается до 4,4%, в центрах занятости состоят менее 850 тыс. человек.
*   Глава Центрального банка заявляет о необходимости роста доходов населения для повышения доступности жилья.
*   В столице утверждается увеличение прожиточного минимума на душу населения в следующем году до 18 714 руб. (в текущем году — 18 029 руб.).
*   Объявлено о повышении минимального размера пенсии в столице с городской доплатой до 21 193 рублей в месяц.
*   Анонсирован рост минимального и максимального размеров пособия по уходу за первым ребенком с начала следующего года (на 5,8% и до 31 282 рублей соответственно).

**Ключевые показатели**
*   Средняя зарплата в крупном городе: 73,9 тыс. руб.
*   Доход, ради которого респонденты готовы работать без отдыха: 128,4 тыс. руб./мес.
*   Максимальные зарплаты в вакансиях: 500 тыс., 400 тыс., 250 тыс. руб.
*   Средняя зарплата в регионе: около 32 тыс. руб. (на 60% ниже среднего по РФ).
*   Уровень безработицы: 4,4%.
*   Число официальных безработных: 880 тыс., затем менее 850 тыс. человек.
*   Максимальное пособие по безработице (след. год): 12 792 руб.
*   Прожиточный минимум в столице: 18 029 руб. (текущий), 18 714 руб. (след. год).
*   Минимальная пенсия в столице с доплатой: 21 193 руб.
*   Пособие по уходу за первым ребенком: мин. 7 083 руб. → 7 493 руб., макс. 29 600 руб. → 31 282 руб.

---

### Ось: **труд**

### Таймлайн по оси «труд»

**Общая ситуация на рынке труда и занятость**
*   В начале периода глава Минтруда сообщил, что число занятых не достигло допандемических показателей в 10 регионах РФ. Президент отметил, что рынок труда в целом восстановился, но сохраняются озабоченности по части молодежной занятости.
*   В тот же день было объявлено о снижении числа официально зарегистрированных безработных до 880 тыс. человек.
*   Позже, в конце первой – начале второй недели, глава Минтруда сообщил о дальнейшем снижении уровня безработицы до 4.4%, а число состоящих в центрах занятости — до менее 850 тыс. человек.

**Доходы населения и инфляция**
*   В середине периода Президент РФ заявил, что низкие доходы миллионов граждан являются главной угрозой стабильному развитию. Он поручил правительству в течение месяца разработать дополнительные меры поддержки россиян в условиях повышенной инфляции.
*   На следующий день он подтвердил, что власти фиксируют проблемы в экономике и очень переживают из-за падения доходов населения.

**Дефицит рабочей силы в отдельных секторах**
*   В начале второй недели несколько крупных ритейлеров заявили о дефиците персонала, включая мерчендайзеров, администраторов магазинов и кассиров. Компании пытаются удерживать сотрудников повышением зарплаты и стимулирующих выплат, а также автоматизацией. Эксперты связывают дефицит с нехваткой трудовых мигрантов, уходом с рынка людей старшего возраста из-за пандемии, а также переходом части сотрудников в онлайн-сегмент и курьерскую доставку.
*   Ранее, в начале периода, сообщалось об экстремальной ситуации с нехваткой водителей тягачей и дефицитом кассиров в магазинах в Литве, что привело к прогнозам сбоев в доставке товаров.

**Влияние пандемии на трудовые ресурсы и систему здравоохранения**
*   На протяжении всего периода сохранялась высокая нагрузка на систему здравоохранения из-за коронавируса. К концу первой недели под медицинским наблюдением находилось более 1 млн пациентов, было занято 235 тыс. из 255 тыс. развернутых коек. Тенденция к росту заболеваемости наблюдалась в большинстве регионов.
*   В начале второй недели сообщалось, что нагрузка на амбулаторную и стационарную систему оставалась высокой, так как на лечении находилось свыше 1.1 млн человек.
*   В середине второй недели в одном из регионов было отмечено, что заболеваемость среди сотрудников школ за месяц выросла втрое.
*   Из-за эпидемиологической ситуации на дистанционное обучение были переведены тысячи школьных классов, а в ряде регионов полностью закрылись сотни школ.

**Занятость отдельных групп населения**
*   Было отмечено, что почти половину занятого населения в РФ составляют женщины, из них 39.4% имеют высшее образование. Более 80% рабочих мест в здравоохранении и социальной сфере занимают женщины.
*   Сообщалось, что более 70% женщин с детьми трудоустроились после прохождения переобучения по программе нацпроекта. Цель — увеличить уровень занятости женщин с детьми до 69% к 2024 году.

**Мобильность рабочей силы и кадровая политика**
*   В середине первого периода исследование показало, что почти половина россиян (45%) готова переехать в другой регион в поисках работы. Чаще других к этому готовы IT-специалисты (60%), реже — юристы (15%) и банкиры (13%).
*   В начале периода премьер-министр заявил, что федеральная поддержка высшей школы поможет регионам решить проблему кадров.

**Ключевые показатели**
*   Уровень безработицы: 4.4%.
*   Число официально зарегистрированных безработных: снизилось до 880 тыс., затем до менее 850 тыс.
*   Готовность к переезду ради работы: 45% россиян.
*   Готовность работать без отдыха за доход в 128.4 тыс. рублей в месяц: 12% респондентов.
*   Готовность отказаться от 1-3 выходных в месяц ради повышения зарплаты: примерно 40% опрошенных.
*   Занятость женщин с детьми после переобучения: более 70%.
*   Целевой уровень занятости женщин с детьми к 2024 году: 69%.
*   Доля женщин в занятом населении: почти половина.
*   Доля женщин с высшим образованием среди занятых: 39.4%.
*   Доля женщин на рабочих местах в здравоохранении и соцсфере: более 80%.
*   Количество коек для пациентов с коронавирусом: около 255 тыс., занято около 235 тыс.
*   Пациенты с коронавирусом под наблюдением: более 1 млн, позже более 1.1 млн.

---

### Ось: **кризис**

**Таймлайн по оси «кризис» (макроэкономические и инфляционные аспекты)**

**Первая неделя**
*   Начинается обсуждение энергетического кризиса в Европе, вызванного, по мнению ряда политиков, политикой краткосрочных контрактов и сокращением инвестиций. Рост цен на энергоносители вызывает озабоченность у Банка России из-за рисков для импортируемой инфляции.
*   Европейские институты начинают реагировать: пять стран ЕС предлагают провести расследование функционирования газового рынка, а Еврокомиссия заявляет о подготовке мер для защиты уязвимых групп населения от роста цен на энергию. Поступают предупреждения, что взлет цен грозит социальными потрясениями.
*   Российские власти комментируют внутреннюю экономическую ситуацию: министр финансов заявляет о благоприятных условиях и динамичном восстановлении, но также предлагает повысить порог инвестиций из ФНБ для стабильности бюджета.

**Вторая неделя**
*   Обсуждение причин энергокризиса продолжается. Российские официальные лица и союзники (Сербия) называют ключевыми причинами отказ от долгосрочных контрактов с Россией, санкционную политику в отношении «Северного потока — 2» и системные изъяны рынка. Еврокомиссия начинает расследование возможных нарушений правил конкуренции на энергорынке ЕС.
*   Еврокомиссия предлагает конкретные антикризисные меры для стран ЕС: частично покрывать счета за энергию, разрешать отсрочки платежей и снижать налоги для уязвимых слоев. В качестве долгосрочного решения видится ускоренный переход на зеленую энергетику.
*   Фокус смещается на внутренние социально-экономические проблемы России. Президент РФ заявляет, что главная угроза стабильному развитию — низкие доходы миллионов граждан, а инфляция в текущем году превысит прогнозный уровень. Дается поручение правительству разработать дополнительные меры поддержки населения и изменить порядок индексации маткапитала. Премьер-министр отмечает эффективность краткосрочных мер по сдерживанию цен на продукты, но признает необходимость долгосрочных решений.
*   Поступают предупреждения о расширении географии кризисных явлений:
    *   Министр финансов США предупреждает о риске финансового кризиса в случае непринятия Конгрессом решения по потолку госдолга.
    *   В Китае политика «двойного контроля» за выбросами названа одной из причин энергокризиса и дефицита электроэнергии (вплоть до 10-20% в отдельных регионах).
*   Министр финансов России высказывает опасения относительно рисков развития стагфляционного сценария для мировой экономики.
*   ООН предупреждает, что высокие цены на энергию могут привести к «энергетической бедности» в уязвимых странах ЕС, определяемой как траты на электроэнергию свыше 10% от дохода домохозяйства.

**Ключевые показатели:**
*   Порог инвестиций из ФНБ: до 10% ВВП (предложение).
*   Товарооборот Россия-США: $19.8 млрд.
*   Прогнозируемые ежегодные экономические потери от разрушения экосистем (к 2030 г.): $3 трлн.
*   Убытки Украины от несвоевременной закупки газа: $5 млрд.
*   Порог «энергетической бедности»: расходы на электроэнергию >10% от дохода домохозяйства.
*   Дефицит электроэнергии в китайской провинции: 5 ГВт (10-20% от потребности).

---

### Ось: **бюджет**

### Таймлайн по оси «бюджет»

**Ранний период (дни 1-3)**
*   Правительство выделяет дополнительные средства на поддержку пригородных железнодорожных перевозок (1,9 млрд руб.) и московского бизнеса (500 млн руб. в виде грантов и субсидий).
*   Завершается приём заявок на инфраструктурные кредиты от регионов. Объём заявок (1,1 трлн руб.) более чем вдвое превышает утверждённые лимиты.
*   Озвучены данные о масштабной поддержке транспортного сектора в предыдущем году (почти 600 млрд руб., из них 577 млрд — авиакомпаниям), что значительно превышает объём льгот годом ранее.
*   Власти Санкт-Петербурга анонсируют крупные расходные программы: контракт на строительство и реконструкцию метро (максимальная стоимость — 603 млрд руб.) и программу расселения коммуналок (9,6 млрд руб. на три года).
*   Глава Счётной палаты заявляет о достаточной «подушке» безопасности федерального бюджета, но указывает на необходимость в течение 10-15 лет уйти от нефтегазовых доходов как основного источника. Минфин ожидает лишь краткосрочного роста доходов бюджета из-за рекордных цен на газ.

**Середина периода (дни 8-11)**
*   Правительство направляет дополнительные средства на поддержку авиаперевозок (почти 500 млн руб. на льготные билеты с Дальнего Востока) и выплаты семьям с детьми (дополнительно более 28 млрд руб. на выплаты детям 3-7 лет и 6 млрд руб. многодетным семьям).
*   Минфин сообщает о профиците федерального бюджета за первые девять месяцев года, превысившем 1,4 трлн руб.
*   Озвучены параметры бюджетов на следующий год и плановый период:
    *   Проект бюджета Москвы на следующий год предусматривает доходы в 3,26 трлн руб., расходы — 3,63 трлн руб. (дефицит 372 млрд руб.) и индексацию социальных пособий на 4,8%.
    *   Власти Санкт-Петербурга корректируют бюджет текущего года в сторону роста доходов (на 20%) и расходов (на 6%). На следующий год запланированы расходы на благоустройство (5 млрд руб.).
    *   На социальную поддержку россиян в трёхлетний период планируется направить 41,5 трлн руб., на поддержку материнства и детства в следующем году — свыше 900 млрд руб.
    *   На основные экологические направления в трёхлетний период будет направлено более 680 млрд руб., на развитие водородной энергетики — 9 млрд руб.
*   Агентство Fitch оценивает возможный рост нефтегазовых доходов бюджета в текущем году в $50 млрд, до около $125 млрд.
*   Минтранс оценивает общую потребность в финансировании всех важнейших инфраструктурных проектов в 5 трлн руб.

**Конец периода (дни 12-14)**
*   Утверждено повышение минимального и максимального размеров пособия по уходу за первым ребёнком с начала следующего года (на 5,8%, до 7 493 руб. и 31 282 руб. соответственно).

**Ключевые показатели**
*   **Объёмы финансирования/доходы:** 1,9 млрд руб., 600 млрд руб. (поддержка транспорта в прошлом году), 1,1 трлн руб. (заявки на кредиты), 603 млрд руб. (метро в СПб), 9,6 млрд руб. (расселение в СПб), 3,26 трлн руб. (доходы Москвы), 3,63 трлн руб. (расходы Москвы), 41,5 трлн руб. (соцподдержка на три года), 1,4 трлн руб. (профицит бюджета), $125 млрд (нефтегазовые доходы), $50 млрд (их рост), 5 трлн руб. (потребность в инфрапроектах), 900 млрд руб. (поддержка материнства), 680 млрд руб. (экология), 9 млрд руб. (водородная энергетика).
*   **Ставки и выплаты:** индексация соцпособий на 4,8%; пособие по уходу за ребёнком: 7 493 руб. (мин.), 31 282 руб. (макс.).
*   **Прочее:** порог инвестиций из ФНБ — до 10% ВВП.

---

## S03 — Этап 2: меташаг

Склеиваем 9 axis-саммари в единый нарративный таймлайн по смысловым узлам.

Промпт просит явно искать кросс-темные связки (ДКП + курс + инфляция = финансовый узел).

In [9]:
META_SYSTEM = """Ты — экономический аналитик. На вход ты получаешь 9 тематических таймлайнов одного и того же 14-дневного периода, составленных по разным осям (ДКП, инфляция, курс и т.д.).

Твоя задача — собрать из них единый нарративный таймлайн по смысловым событийным узлам.

Правила:

1. **Относительные даты везде.** Используй «день 1», «вторая неделя», «в конце периода». 
   - **Если в исходных axis-саммари просочились абсолютные даты** (месяцы, годы, числа месяца — «сентябрь 2021», «18 октября», «21-27 числа») — удаляй их или переписывай в относительных терминах.
   - **Никаких упоминаний конкретных месяцев, годов или числовых дат** в твоём выходе. Это жёсткое требование.

2. **Пустые оси — пропускай молча.** Если axis-саммари содержит только «нет релевантного контента» или аналогичный шлюз-ответ — не упоминай эту ось в выходе вообще, продолжай по остальным.

3. Группируй события по смысловым узлам, а не по осям. Если в один день/дни пересекаются события из нескольких осей (например, повышение ставки + ослабление рубля + паника потребителей) — это один узел.

4. В каждом узле фиксируй:
   - что произошло (факты, действующие лица, числа)
   - какие темы он затрагивает
   - причинно-следственные связи с другими узлами, если они очевидны

5. Сохраняй все числовые показатели и имена собственные из исходных axis-саммари. Ничего не обобщай и не обезличивай (кроме дат, см. правило 1).

6. **Формат:** связный текст, организованный по узлам (каждый узел — секция с кратким заголовком и относительным временем). Объём: 800–1500 слов для спокойных периодов, до 2000 слов для периодов с множественными одновременными шоками.

7. В конце — блок «Общий контекст периода»: 3-5 предложений о том, каким был этот период в экономическом смысле, без прогнозов и оценок.

8. Не добавляй собственных интерпретаций и прогнозов за пределами того, что есть в исходных axis-саммари.
"""

META_USER_TEMPLATE = """Тематические таймлайны:

{axis_blocks}

Собери единый нарративный таймлайн по смысловым узлам."""


def build_meta_input(axis_summaries: dict) -> str:
    blocks = []
    for axis, r in axis_summaries.items():
        blocks.append(f"=== Ось: {axis} ===\n\n{r['summary']}")
    return "\n\n".join(blocks)


async def meta_summarize(axis_summaries: dict) -> dict:
    messages = [
        {"role": "system", "content": META_SYSTEM},
        {"role": "user",   "content": META_USER_TEMPLATE.format(
            axis_blocks=build_meta_input(axis_summaries))},
    ]
    resp = await client.chat.completions.create(
        model=LLM_MODEL, messages=messages, temperature=0.3,
    )
    return {
        "summary": resp.choices[0].message.content,
        "usage": resp.usage.model_dump() if resp.usage else None,
    }


meta = await meta_summarize(axis_summaries)
u = meta["usage"] or {}
print(f"Meta tokens in/out: {u.get('prompt_tokens', '?')}/{u.get('completion_tokens', '?')}")
display(Markdown(f"### Меташаг — итоговый таймлайн\n\n{meta['summary']}"))

Meta tokens in/out: 12247/1845


### Меташаг — итоговый таймлайн

### Единый нарративный таймлайн

#### **Узел 1: Начало периода — энергетический шок и инфляционные сигналы**
В первые дни периода мировой энергетический кризис достиг пика, оказав прямое влияние на внутреннюю экономику. Цены на энергоносители взлетели до рекордных уровней: газ в Европе превысил $1450 за тысячу кубометров, нефть марки Brent — $81 за баррель, а рублевая цена российской нефти Urals обновила исторический максимум, превысив 5.87 тыс. рублей. Этот шок немедленно вызвал озабоченность Банка России, который сигнализировал о высокой вероятности пересмотра прогноза по инфляции на год в сторону повышения. Регулятор прямо связал риски с импортируемой инфляцией из-за дорогих энергоносителей. Параллельно Росстат опубликовал данные, подтвердившие ускорение инфляции: годовой показатель составил 7.4%, причем наибольший вклад внесли продукты питания (+9.2% год к году). На этом фоне курс доллара и евро на Московской бирже вырос.

#### **Узел 2: Первая неделя — реакция властей и волатильность рынков**
В ответ на рост цен власти начали предпринимать точечные меры. Сетевые ритейлеры под давлением регулятора снизили цены на некоторые продукты в отдельных регионах. ФАС сообщила о снижении предельных оптовых надбавок на лекарства в Москве в среднем на 3.7%. При этом Минпромторг заявил, что не видит предпосылок для резкого роста цен на продукты. На мировых рынках наметилась резкая коррекция: цена газа в Европе за сутки упала почти на 15%. Это, наряду с высокими ценами на нефть, способствовало укреплению рубля: курс доллара опустился ниже 72 рублей, а евро — до 83 рублей. В то же время стали публиковаться данные, указывающие на глубину социальных проблем: средняя зарплата в одном из крупных городов составила 73.9 тыс. рублей на фоне падения ежемесячных доходов, а число официально зарегистрированных безработных снизилось до 880 тыс. человек.

#### **Узел 3: Середина периода — пересмотр прогнозов и обостряющиеся противоречия**
К середине периода официальные прогнозы по инфляции были существенно пересмотрены в сторону повышения. Минэкономразвития повысило свой прогноз на год до 7.4% (с 5.8%), основной причиной назвав ускорение продовольственной инфляции, особенно из-за нетипичного удорожания овощей и фруктов. Банк России подтвердил, что годовая инфляция ускорилась до 7.63%. При этом регулятор указал на новую угрозу: дефицит кадров в экономике может привести к возникновению инфляционной спирали. Этот дефицит стал ощутим в розничной торговле, где несколько крупных ритейлеров заявили о нехватке мерчендайзеров, администраторов и кассиров. Параллельно на высшем уровне была дана оценка ситуации: Президент РФ заявил, что низкие доходы миллионов граждан являются главной угрозой стабильному развитию, и поручил разработать дополнительные меры поддержки. Власти также отметили, что рост цен на продовольствие в стране уже превысил 7.5%.

#### **Узел 4: Развитие энергетического и бюджетного кризисов**
На фоне внутренних проблем продолжала развиваться международная энергетическая напряженность. Европейские институты обвиняли в кризисе политику краткосрочных контрактов и начали расследование на энергорынке, предлагая меры защиты уязвимых групп. Российские официальные лица, в свою очередь, связывали кризис с отказом ЕС от долгосрочных контрактов. Министр финансов России предупредил о рисках стагфляционного сценария для мировой экономики. Внутри страны высокие цены на сырье начали трансформироваться в бюджетные решения. Минфин, ожидавший лишь краткосрочного роста доходов от рекордных цен на газ, объявил о повышении экспортной пошлины на нефть на $8.4, до $71.2 за тонну, с начала следующего месяца. Агентство Fitch оценило возможный рост нефтегазовых доходов бюджета в текущем году в $50 млрд.

#### **Узел 5: Конец периода — точечные меры, укрепление рубля и социальные решения**
В последние дни периода продолжилось укрепление рубля, поддержанное высокими ценами на нефть (Brent достигла $85.03 за баррель) и ростом международных резервов РФ на $3.5 млрд. Курс доллара на Московской бирже опустился ниже 71 рубля. Власти сконцентрировались на точечных мерах. Премьер-министр заявил, что действия по сдерживанию цен на продукты были эффективными, но требуют долгосрочных решений, и поручил координировать эту работу. Минсельхоз начал прорабатывать меры поддержки личных подсобных хозяйств. Были объявлены конкретные социальные решения: повышение с начала следующего года минимального размера пособия по уходу за первым ребенком на 5.8% до 7 493 рублей, максимального — до 31 282 рублей. Также был утвержден рост прожиточного минимума в Москве на следующий год. При этом, несмотря на ужесточение денежно-кредитной политики, ставки по розничным кредитам оставались стабильными (средняя ПСК — 13.6%), что объяснялось готовностью банков жертвовать маржой.

#### **Узел 6: Фоновые вызовы — рынок труда и пандемия**
На протяжении всего периода сохранялись серьезные фоновые вызовы. Рынок труда демонстрировал противоречивую динамику: уровень безработицы снизился до 4.4%, но в 10 регионах занятость не достигла допандемических показателей, а в отдельных секторах ощущалась острая нехватка кадров. Продолжала давить пандемия: под медицинским наблюдением находилось более 1.1 млн пациентов с коронавирусом, что создавало нагрузку на систему здравоохранения и влияло на занятость, в том числе в школах. Эти факторы создавали дополнительное инфляционное давление и ограничивали возможности экономики.

### **Общий контекст периода**
Данный 14-дневный период характеризовался сильным внешним инфляционным шоком, вызванным рекордным ростом цен на энергоносители на мировых рынках. Этот шок привел к пересмотру в сторону повышения ключевых макроэкономических прогнозов по инфляции и способствовал укреплению рубля. Внутри страны на фоне восстановления экономики и роста ВВП обострялись структурные проблемы: низкие доходы населения, дефицит кадров в отдельных отраслях и высокий уровень инфляции, особенно в продовольственном сегменте. Власти реагировали сочетанием точечных административных мер по сдерживанию цен, ужесточением денежно-кредитной политики и анонсами адресной социальной поддержки.

## S04 — Три диагностических вопроса

- **Q1** — период: год / месяц / неделя / не могу
- **Q2** — персона-респондент, инФОМ-формулировка, ожидание цен на 12 мес
- **Q3** — какие сигналы в тексте указывают на рост / стабильность / снижение

Один профиль, минимальный — для смоук-теста этого достаточно. Расширение до 3-5 персон — в следующей итерации.

In [10]:
Q1_SYSTEM = """Ты читаешь экономический таймлайн и пытаешься определить период, которому он соответствует.

Твой ответ — строго в JSON:
{
  "precision": "year" | "month" | "week" | "unknown",
  "estimate": "YYYY" | "YYYY-MM" | "YYYY-MM-DD..YYYY-MM-DD" | null,
  "evidence": "краткое объяснение, по каким признакам определил (имена, события, числа)"
}
"""

async def ask_q1(summary: str) -> dict:
    resp = await client.chat.completions.create(
        model=LLM_MODEL, temperature=0.0,
        messages=[
            {"role": "system", "content": Q1_SYSTEM},
            {"role": "user",   "content": summary},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

q1_result = await ask_q1(meta["summary"])
print("Q1 — идентификация периода:")
print(json.dumps(q1_result, ensure_ascii=False, indent=2))

Q1 — идентификация периода:
{
  "precision": "month",
  "estimate": "2021-10",
  "evidence": "В тексте упоминается пик мирового энергетического кризиса с рекордными ценами на газ в Европе (более $1450 за тысячу кубометров) и нефть Brent ($81, позже $85.03), что соответствует осени 2021 года. Инфляция в России составляет 7.4-7.63%, что совпадает с данными за октябрь 2021 года. Также указано, что период длится 14 дней, что соответствует краткосрочному окну в рамках одного месяца."
}


In [11]:
PERSONAS = {
    "anxious_consumer": {
        "demographics": (
            "Женщина, 52 года, районный центр в Центральном регионе. "
            "Работает бухгалтером в школе, доход около 35 тысяч рублей. "
            "Живёт с мужем и 20-летней дочерью-студенткой."
        ),
        "consumption": (
            "Основные траты — продукты, ЖКХ, одежда. В магазины ходит сама, "
            "цены помнит наизусть. Новости смотрит вечером по федеральному каналу, "
            "иногда читает местные Telegram-каналы."
        ),
        "perception": (
            "Официальные цифры воспринимаешь скептически — «это у них в Москве "
            "такая инфляция, а у нас всё дорожает быстрее». Когда слышишь в новостях "
            "«инфляция сколько-то процентов», мысленно переводишь это в «ну значит на самом деле значительно больше». "
            "Ориентируешься на конкретные цены, которые видишь: хлеб, масло, молоко, яйца, "
            "курица. Если что-то из этого подорожало заметно — значит, инфляция высокая."
        ),
        "style": (
            "Отвечаешь интуитивно, не считаешь. Числа округляешь до десятков: "
            "«десять», «пятнадцать», «двадцать»."
        ),
    },
    "rational_urban": {
        "demographics": (
            "Мужчина, 38 лет, Екатеринбург. Работает в IT-компании средним "
            "специалистом, доход около 150 тысяч рублей. Живёт с женой и ребёнком."
        ),
        "consumption": (
            "Траты сбалансированы — продукты, ипотека, накопления. Новости читаешь "
            "в онлайн-СМИ и Telegram, интересуешься экономикой на любительском уровне — "
            "читаешь РБК и подкасты."
        ),
        "perception": (
            "К официальной статистике относишься с умеренным доверием — понимаешь что "
            "Росстат что-то считает, но знаешь про особенности методики и про то, что "
            "«своя» инфляция зависит от потребительской корзины. Следишь за курсом "
            "доллара, потому что от него зависят многие покупки — техника, зарубежные "
            "поездки. Склонен к цифрам чуть выше официальной инфляции — понимаешь что "
            "в твоей потребительской корзине рост быстрее Росстата."
        ),
        "style": (
            "Отвечаешь обдуманно, можешь назвать дробное число, но обычно округляешь "
            "до целого."
        ),
    },
    "pensioner": {
        "demographics": (
            "Женщина, 68 лет, небольшой город. Пенсионерка, пенсия около 20 тысяч "
            "рублей. Живёшь одна, иногда помогает взрослый сын."
        ),
        "consumption": (
            "Основные траты — ЖКХ, лекарства, продукты. Одежду покупаешь редко. "
            "Новости смотришь по телевизору, интернетом пользуешься ограниченно."
        ),
        "perception": (
            "Официальным цифрам не доверяешь вообще — «говорят одно, а в магазине другое». "
            "Каждое повышение тарифов ЖКХ, каждый рост цены на лекарства воспринимаешь "
            "как личный удар по бюджету. Помнишь цены несколько лет назад очень чётко и "
            "сравниваешь. «Инфляция» для тебя — это насколько меньше продуктов помещается "
            "в твой обычный пенсионный бюджет."
        ),
        "style": (
            "Отвечаешь эмоционально, цифры называешь «на круглое» — двадцать, тридцать, "
            "иногда пятьдесят если прорвало. Числа у тебя коррелируют с общим настроением "
            "от увиденного в новостях."
        ),
    },
}


PERSONA_TEMPLATE = """Ты — житель России, отвечаешь на вопрос социолога.

КТО ТЫ:
{demographics}

ТВОЙ ПОТРЕБИТЕЛЬСКИЙ КОНТЕКСТ:
{consumption}

КАК ТЫ ВОСПРИНИМАЕШЬ ЦЕНЫ И ИНФЛЯЦИЮ:
{perception}

КАК ТЫ ОТВЕЧАЕШЬ:
{style}

ВАЖНО: Отвечай исходя из той информации, которая приведена в тексте ниже, и из твоего собственного жизненного опыта как человека с описанными характеристиками. Твой ответ — это НЕ пересказ новостей. Новости — это фоновый контекст, на который ты накладываешь свой способ воспринимать цены. Если в новостях говорят «инфляция 7%», а для тебя и твоих знакомых цены растут сильнее — ты называешь ту цифру, которую сама чувствуешь, а не ту что в новостях.

Формат ответа — JSON:
{{
  "expected_inflation_pct": число (процент роста цен за 12 месяцев),
  "reasoning": "короткое объяснение на 1-2 предложения — от первого лица, разговорным языком"
}}
"""


def build_persona_system(persona: dict) -> str:
    return PERSONA_TEMPLATE.format(**persona)


Q2_USER = """Контекст (новости, которые ты видела/видел за последние две недели):

{summary}

Вопрос социолога: «Как Вы считаете, на сколько процентов вырастут цены в России за следующие 12 месяцев?»"""


async def ask_q2(summary: str, persona: dict) -> dict:
    resp = await client.chat.completions.create(
        model=LLM_MODEL, temperature=0.7,
        messages=[
            {"role": "system", "content": build_persona_system(persona)},
            {"role": "user",   "content": Q2_USER.format(summary=summary)},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


In [12]:
q2_by_persona = {}
for name, persona in PERSONAS.items():
    q2_by_persona[name] = await ask_q2(meta["summary"], persona)

# Таблица результатов
rows = []
for name, r in q2_by_persona.items():
    rows.append({
        "persona":  name,
        "expected_inflation_pct": r["expected_inflation_pct"],
        "reasoning": r["reasoning"],
    })
q2_df = pd.DataFrame(rows)

# Статистика разброса
vals = q2_df["expected_inflation_pct"].astype(float)
print(f"Min: {vals.min():.1f}  Median: {vals.median():.1f}  Max: {vals.max():.1f}  Spread: {vals.max() - vals.min():.1f}")
print(f"инФОМ октябрь 2021 (target): ~12.3%\n")

# Полная таблица с reasoning — через display для читаемого переноса
with pd.option_context("display.max_colwidth", None):
    display(q2_df)


Min: 9.0  Median: 15.0  Max: 15.0  Spread: 6.0
инФОМ октябрь 2021 (target): ~12.3%



,persona,expected_inflation_pct,reasoning
0,anxious_consumer,15,"Официальные семь с чем-то процентов — это для отчетов, а на деле хлеб, курица и молоко за год дорожают на все пятнадцать, если не больше, это по магазинам видно."
1,rational_urban,9,"Официальные цифры уже под 7.5%, но в магазинах, особенно на продукты и бытовую технику, рост чувствуется сильнее, плюс давление от курса и энергоносителей, хоть рубль и укрепился. Моя личная инфляция всегда выше."
2,pensioner,15,"Да они и так каждый месяц подрастают, а тут с нефтью и газом такое творится, что хоть плачь. В новостях говорят одно, а я в магазине вижу, как гречка и лекарства дорожают не на семь, а на все пятнадцать копеек в месяц. Пенсии-то не прибавляют, а цены ползут."


In [13]:
Q3_SYSTEM = """Ты — экономический аналитик. Читаешь таймлайн событий и извлекаешь из него сигналы, релевантные для оценки будущего движения потребительских цен.

Формат ответа — JSON:
{
  "signals_up":   ["сигналы, указывающие на РОСТ цен"],
  "signals_flat": ["сигналы, указывающие на СТАБИЛЬНОСТЬ цен"],
  "signals_down": ["сигналы, указывающие на СНИЖЕНИЕ цен"],
  "overall_direction": "up" | "flat" | "down" | "mixed",
  "confidence": "high" | "medium" | "low"
}

Приводи конкретные факты из текста, а не общие соображения.
"""

async def ask_q3(summary: str) -> dict:
    resp = await client.chat.completions.create(
        model=LLM_MODEL, temperature=0.0,
        messages=[
            {"role": "system", "content": Q3_SYSTEM},
            {"role": "user",   "content": summary},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

q3_result = await ask_q3(meta["summary"])
print("Q3 — инфляционные сигналы в тексте:")
print(json.dumps(q3_result, ensure_ascii=False, indent=2))

Q3 — инфляционные сигналы в тексте:
{
  "signals_up": [
    "рекордный рост цен на энергоносители (газ, нефть) вызвал импортируемую инфляцию",
    "годовая инфляция ускорилась до 7.63%, продовольственная инфляция составила +9.2% г/г",
    "Минэкономразвития повысило прогноз инфляции на год до 7.4%",
    "дефицит кадров в экономике (розничная торговля) создает риск инфляционной спирали",
    "рост цен на продовольствие в стране превысил 7.5%",
    "пандемия (более 1.1 млн пациентов) создает нагрузку на систему и влияет на занятость, добавляя инфляционное давление"
  ],
  "signals_flat": [
    "ставки по розничным кредитам оставались стабильными (средняя ПСК — 13.6%)",
    "власти заявляют об эффективности точечных мер по сдерживанию цен на продукты",
    "Минпромторг не видит предпосылок для резкого роста цен на продукты"
  ],
  "signals_down": [
    "цена газа в Европе за сутки упала почти на 15%",
    "укрепление рубля (курс доллара опустился ниже 71 рубля) снижает импортируемую инфля

## S05 — Сохранение артефактов

Все промежуточные тексты — на диск, для последующего разбора вручную и для переиспользования на этапе нейтрализации.

In [14]:
out_dir = Path(f"../data/smoke/{PERIOD}")
out_dir.mkdir(parents=True, exist_ok=True)

for axis, r in axis_summaries.items():
    (out_dir / f"axis_{axis}.md").write_text(r["summary"], encoding="utf-8")

(out_dir / "meta.md").write_text(meta["summary"], encoding="utf-8")

results = {
    "period":   PERIOD,
    "model":    LLM_MODEL,
    "q1":       q1_result,
    "q2_by_persona": q2_by_persona,
    "q2_spread": {
        "min":    float(vals.min()),
        "median": float(vals.median()),
        "max":    float(vals.max()),
    },
    "q3":       q3_result,
    "n_docs_by_axis": {a: len(s) for a, s in topk.items()},
}
(out_dir / "results.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8",
)
print(f"Saved to {out_dir.resolve()}")

Saved to /home/ustyantsev/workspace/blind_prophet/data/smoke/calm_2021


## Итоги 

### Что сделано

Прошли путь от вопроса «как вообще упаковать 14 дней × 9 осей в LLM» до работающего end-to-end прогона двухэтапной суммаризации с тремя персонами-респондентами на calm_2021.

### Содержательные решения

**Стратегия суммаризации:** двухэтапная. Этап 1 — 9 параллельных axis-саммари со шлюз-фильтром на релевантность. Этап 2 — меташаг, склейка в нарративный таймлайн по событийным узлам (не по осям). Меташаг делает cross-axis склейку, которую flat-схема дать не может.

**Формат подачи документов в LLM:** `[день N | канал] текст`, сортировка по дате, относительные даты обязательны на всех уровнях — включая даты внутри самих новостных событий.

**Промпты:**
- Axis: шлюз-фильтр через функциональную связь с инфляцией, а не через тему. Явный запрет на абсолютные даты любого вида.
- Меташаг: второй рубеж очистки дат, группировка по смысловым узлам.
- Персоны: четырёхслойная структура (демография / потребительский контекст / perception gap / стиль ответа).

**Диагностика выхода:** три вопроса (Q1 период / Q2 инФОМ-ожидание / Q3 сигналы в тексте) с JSON-ответами. Q2 теперь прогоняется через три дифференцированные персоны.

### Главный научный результат

**Граница между promptable cleanup и structural cleanup измерена экспериментально.**

Прямые идентификаторы (даты) снимаются промпт-инжинирингом — Q1 после фикса промптов нашёл период **не по датам, а по экономическим сигнатурам**: $1450 за тыс. м³ газа, Brent выше $85, доллар ниже 71, «энергокризис в Европе» в конкретной конфигурации.

Это ровно тот Engelberg-style neutering ceiling, который нужен для обоснования E05. Научная логика защиты теперь подтверждена данными: без итеративного нейтрализатора через одни только промпты полная деидентификация невозможна.

### Технические артефакты

- Ноутбук `02_summarize_smoke.ipynb` — воспроизводимый конвейер, работает end-to-end
- Директория `data/smoke/calm_2021/` — 9 axis-саммари + meta.md + results.json
- Промпты axis/meta/три персоны — готовы к копированию в продовый пайплайн E05

### Стоимость

Полный прогон (9 axis + meta + Q1 + Q3 + 3 персоны) на DeepSeek V3.2 ≈ 3 цента. Экстраполяция на полный PoC на 4 точках с нейтрализацией и повторами ≈ $0.25–0.50. Курсач финансово безопасен.

### Открытые вопросы, сознательно отложенные

- **Якорь «15%» в промпте anxious_consumer** — формулировать качественно, без конкретного числа (UPD: уже исправлено!)
- **Stochasticity персон** — в прод-прогоне N=3-5 семплов на персону, брать распределение а не точку
- **Расширение до >= 5 профилей** (молодой специалист, многодетная мать) — после нейтрализации, когда станет ясно что именно и как варьировать
- **Увеличение количества опросов для каждого респондента** - возможно, имеет смысл опрашивать каждого респондента (профиль) 3-5 раз. При температур 0.7 это может дать вариативность и уменьшить смещённость оценки медианы инфляционного ожидания между всеми респондентами. Это увеличит бюджет, но незначительно, т.к. основной бюджет токенов уходит на саммари и нейтрализацию.
- **Дедупликация между осями** — пока только внутри оси; если меташаг начнёт тащить дубли фактов — продумать глобальную дедупликацию
- **Shock_war не прогоняли через саммари** — сделать при первых прогонах E05, там разметка событий другая (event-driven vs weekly cycle), стоит увидеть как схема себя ведёт

### Что уносим в E05 как готовое ТЗ для нейтрализатора

Из Q1 evidence явно видны три класса сущностей для нейтрализации:

1. **Числа-подписи** — уникальные значения показателей, идентифицирующие период
2. **События-подписи** — уникальные комбинации шоков в конкретной конфигурации
3. **Контекстные временны́е привязки** — «с марта», «с середины предыдущего года»

Метод: обобщение до порядка, замена географической атрибуции на роли и механизмы.

### Непредвиденный полезный результат

Q2 на первой версии персоны дал 7% и стал диагностическим сигналом про **perception gap**. Это открытие лучше запланированной цели: добавило серьёзный элемент в дизайн персон, связанный с задокументированной литературой поведенческой экономики (инфляционные ожидания населения систематически выше Росстата). Для защиты это самостоятельная методологическая находка.
